# 02 · Token-size check on Beirut (decides ADR 0001)

**Task 6 of the next-steps guide.** How many ViT tokens does each building cover? If most buildings
cover fewer than ~4 tokens, pooling mixes building and ground, and small C2 shifts may not change any token.

- Buildings here = connected components of the Beirut label raster (label > 0). Touching buildings can merge,
  so small buildings are, if anything, **under**-counted; the instance JSON gives the exact answer later.
- Beirut is a development event, so reading its labels is allowed.
- The resolution table at the end reads only geotransforms (no labels), so it covers all 14 events.

In [ ]:
NOTEBOOK_NAME = "02_token_size_beirut"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Download BRIGHT SAR + labels from Hugging Face straight to /content (~3.4 GB, ~1 min).
# Copying from Drive fails (Errno 107), so always download fresh.
import os, zipfile
from huggingface_hub import hf_hub_download
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # BRIGHT is not gated; the token only avoids rate limits

ROOT = "/content/bright"
os.makedirs(ROOT, exist_ok=True)
for name in ["post-event.zip", "target.zip"]:
    folder = os.path.join(ROOT, name.replace(".zip", ""))
    if os.path.isdir(folder) and len(os.listdir(folder)) > 4000:
        print("already there:", folder); continue
    p = hf_hub_download("Kullervo/BRIGHT", name, repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
    print("unzipped", name)
for sub in ["post-event", "target"]:
    print(sub, len(os.listdir(os.path.join(ROOT, sub))), "files")

In [ ]:
import glob, os
import numpy as np, pandas as pd, rasterio
from scipy import ndimage

PATCH = 16                     # ViT-B/16 (DINOv3, ImageNet ViT). DINOv2 uses 14.
TARGET_RES = 0.5               # candidate common ground resolution (m/pixel)

rows = []
for lab_path in sorted(glob.glob(f"{ROOT}/target/beirut-*_building_damage.tif")):
    sar_path = lab_path.replace("/target/", "/post-event/").replace("_building_damage", "_post_disaster")
    with rasterio.open(lab_path) as src:
        lab = src.read(1)
        res = abs(src.transform.a)            # metres per pixel (projected CRS)
    with rasterio.open(sar_path) as s:
        nod = (s.read(1) == 0)
    if nod.mean() >= 0.95:
        continue                               # no-data rule: tile excluded
    comp, n = ndimage.label(lab > 0)
    if n == 0:
        continue
    area_px = ndimage.sum(np.ones_like(comp), comp, index=np.arange(1, n + 1))
    nod_frac = ndimage.mean(nod, comp, index=np.arange(1, n + 1))
    cls = ndimage.maximum(lab, comp, index=np.arange(1, n + 1))
    for a, f, c in zip(area_px, nod_frac, cls):
        if f >= 0.5:
            continue                           # no-data rule: building dropped
        rows.append({"tile": os.path.basename(lab_path), "res_m": res, "area_px": a,
                     "area_m2": a * res * res, "label": int(c)})

b = pd.DataFrame(rows)
b["tokens_native"] = b["area_px"] / PATCH**2
b["tokens_at_target"] = b["area_m2"] / (PATCH * TARGET_RES) ** 2
print(len(b), "buildings in", b["tile"].nunique(), "tiles; resolution(s):", sorted(b["res_m"].round(3).unique()))
print("token side at native resolution:", (PATCH * b["res_m"]).round(2).unique(), "m")

In [ ]:
def shares(col):
    return {f"< {k} tokens": round((b[col] < k).mean() * 100, 1) for k in (1, 4, 9)}

summary = pd.DataFrame({
    "native": shares("tokens_native"),
    f"{TARGET_RES} m": shares("tokens_at_target"),
}).T
print("Share of Beirut buildings (%) covering fewer than k tokens")
display(summary)
print("Median tokens per building: native",
      round(b["tokens_native"].median(), 2), "|", TARGET_RES, "m", round(b["tokens_at_target"].median(), 2))
print("Median footprint area:", round(b["area_m2"].median(), 1), "m2")

In [ ]:
# Does a C2 shift of d metres move a footprint by at least one token?
token_side_native = float((PATCH * b["res_m"]).median())
token_side_target = PATCH * TARGET_RES
for d in [2, 5, 10, 15, 27]:
    print(f"shift {d:>2} m = {d / token_side_native:4.2f} tokens native | {d / token_side_target:4.2f} tokens at {TARGET_RES} m")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.logspace(-2, 3, 60)
ax.hist(b["tokens_native"], bins=bins, alpha=0.6, label="native")
ax.hist(b["tokens_at_target"], bins=bins, alpha=0.6, label=f"{TARGET_RES} m")
for k in (1, 4, 9):
    ax.axvline(k, color="k", lw=0.8, ls="--")
ax.set_xscale("log"); ax.set_xlabel("tokens per building (ViT, 16 px patches)"); ax.set_ylabel("buildings")
ax.set_title("Beirut: building size in tokens"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Resolution per event: geotransforms only, no labels read (safe for confirmatory events)
import math
res_rows = []
for p in sorted(glob.glob(f"{ROOT}/post-event/*_post_disaster.tif")):
    ev = os.path.basename(p).split("-", 1)[0].lower()
    with rasterio.open(p) as s:
        a = abs(s.transform.a)
        if s.crs and s.crs.is_geographic:          # EPSG:4326 tiles (Bata, Congo, Haiti, Noto): degrees -> metres
            lat = (s.bounds.top + s.bounds.bottom) / 2
            a = a * 111320 * math.cos(math.radians(lat))
        res_rows.append({"event": ev, "crs": s.crs.to_string() if s.crs else None, "res_m": round(a, 3),
                         "size": f"{s.width}x{s.height}"})
r = pd.DataFrame(res_rows)
res_table = r.groupby("event").agg(tiles=("res_m", "size"), res_min=("res_m", "min"), res_max=("res_m", "max"),
                                   crs=("crs", "first"))
res_table["token_side_native_m"] = (res_table["res_max"] * PATCH).round(1)
display(res_table)

In [ ]:
# Save results for the ADR into this run's folder
summary.to_csv(f"{RUN}/beirut_token_shares.csv")
res_table.to_csv(f"{RUN}/resolution_by_event.csv")
plt.figure(fig.number); fig.savefig(f"{RUN}/beirut_tokens.png", dpi=120)
print("saved to", RUN)

**Decide ADR 0001** (`docs/adr/0001-resolution-and-pooling.md`):

- If < 20% of buildings are under 4 tokens at the chosen resolution → area-weighted pooling at that resolution is enough.
- If 20–50% → add ×2 bilinear upsampling of the token grid before pooling.
- If > 50% → consider building-centred crops (check the time budget first).

These cut-offs are a suggestion; write the one you choose into the ADR **before** computing any AUROC.